Ranking the hosts of simulated BBH events.

The code was written by Mária Pálfi (marika97@student.elte.hu).

In [ ]:
# importing useful packages and functions
import healpy as hp
import numpy as np
from matplotlib import pyplot as plt
from scipy.stats import norm
import pandas as pd
import json
from astropy import units as u
import astropy_healpix as ah
from ligo.skymap.io import read_sky_map

Reading weights from Artale 

In [ ]:
prob_M_BBH = pd.read_csv('Probability_Mstar_BBH_z0p1.dat', delimiter='\t', 
                         header = 0, names = ['logMs', 'P'] )

Defining functions for ranking:

In [ ]:
def find_nearest(array, value, return_array):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return return_array[idx]

def dp_dV_calc(hosts, n, dp_dV, nside, ipix, prob, distmu, distsigma, distnorm, pixarea):
    match_ipix = ah.lonlat_to_healpix( hosts.iloc[n]['ra']*180/np.pi*u.deg, 
                                      hosts.iloc[n]['dec']*180/np.pi*u.deg,
                                      nside, order = 'nested'  )
    i = np.flatnonzero(ipix == match_ipix)[0]
    dp_dV.append( prob[i] * distnorm[i] * 
                     norm( distmu[i], distsigma[i] ).pdf(hosts.iloc[n]['dL']) 
                     / pixarea[i] )
    
def pM_Ducoin( hosts, M, dp_dV ):
    pM = M/np.sum(M)
    hosts['dp_dV_pM'] = hosts.dp_dV*pM
    return hosts.sort_values( by = ['dp_dV_pM'] , axis = 0, ascending = False ).index

def pM_Artale( hosts, lg_sm, dp_dV ):
    prob_M_vals = []
    for i in range(len(lg_sm)):
        prob_M_vals.append( find_nearest( prob_M_BBH['logMs'], lg_sm[i], prob_M_BBH['P'] ) )
    prob_M_Artale = prob_M_vals/np.sum(prob_M_vals)
    hosts['prob_M_Artale'] = prob_M_Artale
    hosts['prob_Artale'] = hosts.prob_M_Artale * dp_dV
    return hosts.sort_values( by = ['prob_Artale'] , axis = 0, ascending = False ).index

Defining functions for stellar mass estimation:

In [ ]:
def abs_magni( m, dL ):
    return m - 5 * np.log10( dL ) - 25

def luminosity_W1( M1 ):
    return 10**( -0.4 * ( M1 - 3.24 ) ) #M_sun = 3.24

def M_Cluver( M1, M2, L_W1 ):
    TM = -1.96*( M1 - M2 ) - 0.03
    return L_W1 * 10**TM

def M_Jarrett( M1, M2, L_W1 ):
    TM = -0.246 - 2.1 *( M1 - M2 ) 
    return L_W1*10**(TM)

def M_Kettlety( L_W1 ):
    return 10**(-0.4)*L_W1

def M_Cappellari( M_K ):
    return 10**( 10.58 - 0.44 * (M_K + 23) ) 

Function for ranking the host of one event:

In [ ]:
def sorting_one_event( num, just_one = False ):
    print( "Event ", num )
    if num in [133]:
        print( 'Skip event' + str(num) + '. Too small localization area!' )
    else:
        # Real host's coordinates
        f = open('injections/injection' + str(num) + '.json')
        data = json.load(f)
        real_ra = data['injections']['content']['ra'][0]
        real_dec = data['injections']['content']['dec'][0]
        real_dL = data['injections']['content']['luminosity_distance'][0]
        f.close()
        
        # Read-in possible hosts
        hosts = pd.read_csv('host_files/host' + str(num) + '.txt')
        if np.sum(  hosts['ra'] == real_ra ) == 0:
            print( 'Skip event' + str(num) +
                  '. The real host is not in the 90% localization volume!' )
            not_in.append( num )

        elif np.sum(  hosts['dec'] == real_dec ) == 0:
            print( 'Skip event' + str(num) + 
                  '. The real host is not in the 90% localization volume!' )
            not_in.append( num )

        elif np.sum(  hosts['dL'] == real_dL ) == 0:
            print( 'Skip event' + str(num) + 
                  '. The real host is not in the 90% localization volume!' )
            not_in.append( num )

        else:
            # Where is the real host
            host_ra_idx = hosts.index[ hosts['ra'] == real_ra ][0]
            host_dec_idx = hosts.index[ hosts['dec'] == real_dec ][0]
            idx = hosts.index[ hosts['dL'] == real_dL ][0]
            if host_ra_idx == host_dec_idx & host_dec_idx == idx:
            
                hosts['M_B'] = abs_magni( hosts.m_B, hosts.dL )
                hosts['M_K'] = abs_magni( hosts.m_K, hosts.dL )
                hosts['M_W1'] = abs_magni( hosts.m_W1, hosts.dL )
                hosts['M_W2'] = abs_magni( hosts.m_W2, hosts.dL )
                # Calculating W1 band luminosity
                hosts['L_W1'] = luminosity_W1( hosts.M_W1 )
                # Calculating absolute magnitudes
                hosts['Jarrett'] = M_Jarrett( hosts.M_W1, hosts.M_W2, hosts.L_W1  )
                hosts['Cluver'] = M_Cluver( hosts.M_W1, hosts.M_W2, hosts.L_W1  )
                hosts['Kettlety'] = M_Kettlety( hosts.L_W1 )
                hosts['Cappellari'] = M_Cappellari( hosts.M_K )

                if just_one == True:
                    filename = "skymaps/skymap"+str(num)+".fits"
                else:
                    filename = './outdir_folder/bbh_injection_'+str(num)+'/result/bbh_injection_data0_0_analysis_H1L1V1_skymap.fits'
                skymap = read_sky_map( filename, moc = True )
                uniq = skymap['UNIQ']
                prob = skymap['PROBDENSITY']
                distnorm = skymap['DISTNORM']
                distmu = skymap['DISTMU']
                distsigma = skymap['DISTSIGMA']

                level, ipix = ah.uniq_to_level_ipix(uniq)
                nside = ah.level_to_nside(level)
                pixarea = hp.nside2pixarea(nside)

                dp_dV = []
                for n in range(len(hosts)): dp_dV_calc(hosts, n, dp_dV, nside, ipix,
                                                       prob, distmu, distsigma, distnorm, pixarea)
                hosts['dp_dV'] = dp_dV
                s_dp_dV = hosts.sort_values( by = ['dp_dV'] , axis = 0, ascending = False ).index
                s_dp_dV_list.append( np.argwhere( s_dp_dV == idx )[0][0] )
                
                s_Ducoin = pM_Ducoin( hosts, hosts.stellar_mass, dp_dV )
                s_Ducoin_list.append( np.argwhere( s_Ducoin == idx )[0][0] )
                s_Artale = pM_Artale( hosts, np.log10(hosts.stellar_mass*0.73)+10, dp_dV )
                s_Artale_list.append( np.argwhere( s_Artale == idx )[0][0] )

                s_D_Jarrett = pM_Ducoin( hosts, hosts.Jarrett, dp_dV )
                s_D_Jarrett_list.append( np.argwhere( s_D_Jarrett == idx )[0][0] )
                s_A_Jarrett = pM_Artale( hosts, np.log10(hosts.Jarrett), dp_dV )
                s_A_Jarrett_list.append( np.argwhere( s_A_Jarrett == idx )[0][0] )

                s_D_Cluver = pM_Ducoin( hosts, hosts.Cluver, dp_dV )
                s_D_Cluver_list.append( np.argwhere( s_D_Cluver == idx )[0][0] )
                s_A_Cluver = pM_Artale( hosts, np.log10(hosts.Cluver), dp_dV )
                s_A_Cluver_list.append( np.argwhere( s_A_Cluver == idx )[0][0] )

                s_D_Kettlety = pM_Ducoin( hosts, hosts.Kettlety, dp_dV )
                s_D_Kettlety_list.append( np.argwhere( s_D_Kettlety == idx )[0][0] )
                s_A_Kettlety = pM_Artale( hosts, np.log10(hosts.Kettlety), dp_dV )
                s_A_Kettlety_list.append( np.argwhere( s_A_Kettlety == idx )[0][0] )

                s_D_Cappellari = pM_Ducoin( hosts, hosts.Cappellari, dp_dV )
                s_D_Cappellari_list.append( np.argwhere( s_D_Cappellari == idx )[0][0] )
                s_A_Cappellari = pM_Artale( hosts, np.log10(hosts.Cappellari), dp_dV )
                s_A_Cappellari_list.append( np.argwhere( s_A_Cappellari == idx )[0][0] )
                
                length.append( len(hosts) )

                if just_one == True:
                    return s_dp_dV, s_Ducoin, s_Artale, s_D_Jarrett, s_A_Jarrett, s_D_Cluver, s_A_Cluver, s_D_Kettlety, s_A_Kettlety, s_D_Cappellari, s_A_Cappellari
            else:
                print( 'Skip event' + str(num) +
                      '. The real host is not in the 90% localization volume!' )
                not_in.append( num )

In [ ]:
# empty lists to save results
not_in, length = [], []
s_dp_dV_list, s_Ducoin_list, s_Artale_list = [], [], []
s_D_Jarrett_list, s_D_Cluver_list, s_D_Kettlety_list, s_D_Cappellari_list = [], [], [], [] 
s_A_Jarrett_list, s_A_Cluver_list, s_A_Kettlety_list, s_A_Cappellari_list = [], [], [], [] 

In [ ]:
for e in range(195): sorting_one_event( e )

In [ ]:
len(s_dp_dV_list), len(not_in)

Fraction when stellar mass is improving the ranking:

In [ ]:
def fraction( listname ):
    filt = np.array(s_dp_dV_list) - np.array(listname) > 0 
    print( np.round(len(np.array(s_dp_dV_list)[filt])/len(np.array(s_dp_dV_list))*100) )

In [ ]:
print( 'Ducoin with true value' )
fraction( s_Ducoin_list )
print( 'Artale with true value'  )
fraction( s_Artale_list )

print( 'Ducoin with Jarrett' )
fraction( s_D_Jarrett_list )
print( 'Artale with Jarrett' )
fraction( s_A_Jarrett_list )

print( 'Ducoin with Cluver' )
fraction( s_D_Cluver_list )
print( 'Artale with Cluver' )
fraction( s_A_Cluver_list )

print( 'Ducoin with Kettlety' )
fraction( s_D_Kettlety_list,  )
print( 'Artale with Kettlety' )
fraction( s_A_Kettlety_list)

print( 'Ducoin with Cappellari' )
fraction( s_D_Cappellari_list )
print( 'Artale with Cappellari' )
fraction( s_A_Cappellari_list )

Normalized rank difference:

In [ ]:
def norm_rank_diff( listname ):
    return (np.array(s_dp_dV_list) - np.array(listname) ) / (length )

Bootstrapping to calculate error fro mean and standard deviation of normalized rank difference:

In [ ]:
def bootstrapping( listname, nit = 10000, alpha = 0.95 ):
    nrd_arr = norm_rank_diff( listname )
    means = []
    stds = []
    
    for i in range( nit ):
        sample = np.random.choice( nrd_arr, size = len(nrd_arr) )
        means.append( np.mean( sample ) )
        stds.append( np.std( sample ) )
     
    p = ((1.0-alpha)/2.0) * 100
    lower = max(0.0, np.percentile(means, p))
    lower_s = max(0.0, np.percentile(stds, p))  
    p = (alpha+((1.0-alpha)/2.0)) * 100
    upper = min(1.0, np.percentile(means, p))
    upper_s = min(1.0, np.percentile(stds, p))
    M = np.mean(means)
    S = np.mean(stds)
    print( np.round(M, decimals = 3 ), np.round( M-lower, decimals = 3), 
          np.round( upper-M, decimals = 3 ) )
    print( np.round( S, decimals = 3 ), np.round( S-lower_s, decimals = 3 ),
          np.round( upper_s-S, decimals = 3 ) )

In [ ]:
print( 'Ducoin' )
bootstrapping( s_Ducoin_list )

print( 'Artale' )
bootstrapping( s_Artale_list )

print( 'Ducoin with Jarrett' )
bootstrapping( s_D_Jarrett_list )

print( 'Artale with Jarrett' )
bootstrapping( s_A_Jarrett_list )

print( 'Ducoin with Cluver' )
bootstrapping( s_D_Cluver_list )

print( 'Artale with Cluver' )
bootstrapping( s_A_Cluver_list )

print( 'Ducoin with Kettlety' )
bootstrapping( s_D_Kettlety_list )

print( 'Artale with Kettlety' )
bootstrapping( s_A_Kettlety_list )

print( 'Ducoin with Cappellari' )
bootstrapping( s_D_Cappellari_list )

print( 'Artale with Cappellari' )
bootstrapping( s_A_Cappellari_list )

### Plotting the distribution of normalized rank differences

with boxplots:

In [ ]:
x = [ norm_rank_diff( s_Ducoin_list ),
      norm_rank_diff( s_D_Jarrett_list ),
      norm_rank_diff( s_D_Cluver_list ),
      norm_rank_diff( s_D_Kettlety_list ),
      norm_rank_diff( s_D_Cappellari_list ),
    ]

labels = ['Simulation', 'Jarrett', 'Cluver', 'Kettlety', 'Cappellari' ]

fig, ax = plt.subplots(nrows = 2)
ax[0].set_ylabel('Ducoin')
bplot = ax[0].boxplot( x,  labels=labels )
    
x = [ norm_rank_diff( s_Artale_list ),
      norm_rank_diff( s_A_Jarrett_list ),
      norm_rank_diff( s_A_Cluver_list ),
      norm_rank_diff( s_A_Kettlety_list ),
      norm_rank_diff( s_A_Cappellari_list ),
    ]

ax[1].set_ylabel('Artale')
bplot = ax[1].boxplot( x, labels=labels)  # will be used to label x-ticks

ax[0].yaxis.grid()
ax[1].yaxis.grid()
plt.show()

with violinplots:

In [ ]:
x = [ norm_rank_diff( s_Ducoin_list ),
      norm_rank_diff( s_D_Jarrett_list ),
      norm_rank_diff( s_D_Cluver_list ),
      norm_rank_diff( s_D_Kettlety_list ),
      norm_rank_diff( s_D_Cappellari_list ),
    ]

labels = ['Simulation', 'Jarrett', 'Cluver', 'Kettlety', 'Cappellari' ]

fig, ax = plt.subplots(nrows = 2)
ax[0].set_ylabel('Ducoin')
bplot = ax[0].violinplot( x )
    
x = [ norm_rank_diff( s_Artale_list ),
      norm_rank_diff( s_A_Jarrett_list ),
      norm_rank_diff( s_A_Cluver_list ),
      norm_rank_diff( s_A_Kettlety_list ),
      norm_rank_diff( s_A_Cappellari_list ),
    ]

ax[1].set_ylabel('Artale')
bplot = ax[1].violinplot( x )  # will be used to label x-ticks

ax[0].yaxis.grid()
ax[1].yaxis.grid()
ax[0].set_xticks([y + 1 for y in range(len(x))],
                  labels=labels)
ax[1].set_xticks([y + 1 for y in range(len(x))],
                  labels=labels)
plt.savefig('nrd.png', dpi = 100)
plt.show()

### Normalized rank

In [ ]:
def rank( listname ):
    return ( np.array(listname) ) / (length )

Boxplot:

In [ ]:
x = [ rank( s_dp_dV_list ),
      rank( s_Ducoin_list ),
      rank( s_D_Jarrett_list ),
      rank( s_D_Cluver_list ),
      rank( s_D_Kettlety_list ),
      rank( s_D_Cappellari_list ),
    ]

labels = ['Localization', 'Simulation', 'Jarrett', 'Cluver', 'Kettlety', 'Cappellari' ]

fig, ax = plt.subplots(nrows = 2)
ax[0].set_ylabel('Ducoin')
bplot = ax[0].boxplot( x,  labels=labels )
    
x = [ rank( s_dp_dV_list ),
      rank( s_Ducoin_list ),
      rank( s_D_Jarrett_list ),
      rank( s_D_Cluver_list ),
      rank( s_D_Kettlety_list ),
      rank( s_D_Cappellari_list ),
    ]

ax[1].set_ylabel('Artale')
bplot = ax[1].boxplot( x, labels=labels)  # will be used to label x-ticks

ax[0].yaxis.grid()
ax[1].yaxis.grid()
plt.show()

Violinplot:

In [ ]:
x = [ rank( s_dp_dV_list ),
      rank( s_Ducoin_list ),
      rank( s_D_Jarrett_list ),
      rank( s_D_Cluver_list ),
      rank( s_D_Kettlety_list ),
      rank( s_D_Cappellari_list ),
    ]

labels = ['Localization', 'Simulation', 'Jarrett', 'Cluver', 'Kettlety', 'Cappellari' ]

fig, ax = plt.subplots(nrows = 2)
ax[0].set_ylabel('Ducoin')
bplot = ax[0].violinplot( x )
plt.gca().invert_yaxis()
#ax[0].set_yscale('log')
    
x = [ rank( s_dp_dV_list ),
      rank( s_Ducoin_list ),
      rank( s_D_Jarrett_list ),
      rank( s_D_Cluver_list ),
      rank( s_D_Kettlety_list ),
      rank( s_D_Cappellari_list ),
    ]

ax[1].set_ylabel('Artale')
bplot = ax[1].violinplot( x )  # will be used to label x-ticks

ax[0].yaxis.grid()
ax[1].yaxis.grid()
ax[0].set_xticks([y + 1 for y in range(len(x))],
                  labels=labels)
ax[1].set_xticks([y + 1 for y in range(len(x))],
                  labels=labels)
ax[0].yaxis.set_inverted(True)
#plt.yscale('log')
plt.savefig('rank.png', dpi = 100)
plt.show()